# Metrica Tracking Data → Player Graph

Loads Metrica Sports Sample Game 1 tracking data and builds spatial player graphs  
using `src/graph_builder.py` and `src/features.py`.

**Graph structure**
- Nodes  — players (features: x, y, vx, vy, team)
- Edges  — Delaunay triangulation of player positions  
- Labels — pass outcome from events data (1 = successful pass)

**Sections**
1. Load & parse tracking CSV
2. Extract a single frame → build graph
3. Enrich with distance/pressure/angle features
4. Visualise graph on pitch
5. Build a dataset of graphs aligned to pass events

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../.."))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import torch
from mplsoccer import Pitch

from src.graph_builder import (
    build_graph_from_tracking_frame,
    EdgeStrategy,
    PITCH_LENGTH,
    PITCH_WIDTH,
)
from src.features import enrich_graph

## 1. Load & Parse Metrica Tracking Data

In [ ]:
DATA_DIR = "../../data/raw/metrica/data/Sample_Game_1"
GAME     = "Sample_Game_1"


def load_tracking(filepath: str, team: str) -> pd.DataFrame:
    """
    Parse a Metrica tracking CSV into a tidy DataFrame.

    Returns columns:
      period, frame, time, player, jersey, x, y
    where x,y are in metres (scaled from [0,1]).
    """
    raw = pd.read_csv(filepath, header=[0, 1, 2])

    # The first 3 columns are Period, Frame, Time
    meta_cols = raw.columns[:3]
    period = raw[meta_cols[0]].values.astype(int)
    frame  = raw[meta_cols[1]].values.astype(int)
    time   = raw[meta_cols[2]].values.astype(float)

    records = []
    # Player columns come in x/y pairs: (Team, Jersey, PlayerN) then unnamed
    col_list = raw.columns.tolist()
    i = 3
    while i < len(col_list):
        lvl0, lvl1, lvl2 = col_list[i]
        if team in str(lvl0) and "Player" in str(lvl2):
            jersey = str(lvl1)
            player = str(lvl2)
            x_vals = raw[col_list[i]].values.astype(float)
            y_vals = raw[col_list[i + 1]].values.astype(float)
            for f_idx in range(len(frame)):
                if not (np.isnan(x_vals[f_idx]) or np.isnan(y_vals[f_idx])):
                    records.append({
                        "period": period[f_idx],
                        "frame":  frame[f_idx],
                        "time":   time[f_idx],
                        "player": player,
                        "jersey": jersey,
                        "x":      x_vals[f_idx] * PITCH_LENGTH,
                        "y":      y_vals[f_idx] * PITCH_WIDTH,
                    })
            i += 2
        else:
            i += 1

    return pd.DataFrame(records)


print("Loading tracking data...")
home_tracking = load_tracking(f"{DATA_DIR}/{GAME}_RawTrackingData_Home_Team.csv", "Home")
away_tracking = load_tracking(f"{DATA_DIR}/{GAME}_RawTrackingData_Away_Team.csv", "Away")

print(f"Home: {home_tracking['player'].nunique()} players, {home_tracking['frame'].nunique():,} frames")
print(f"Away: {away_tracking['player'].nunique()} players, {away_tracking['frame'].nunique():,} frames")
home_tracking.head(3)

In [ ]:
def load_events(filepath: str) -> pd.DataFrame:
    """Load Metrica events CSV."""
    events = pd.read_csv(filepath)
    events.columns = [c.strip().lower().replace(" ", "_").replace("[", "").replace("]", "") for c in events.columns]
    return events


events = load_events(f"{DATA_DIR}/{GAME}_RawEventsData.csv")
print(f"{len(events)} events, types: {events['type'].unique()}")
events.head(5)

## 2. Compute Velocities & Extract a Single Frame

In [ ]:
FPS = 25.0  # Metrica tracking frame rate


def add_velocities(df: pd.DataFrame) -> pd.DataFrame:
    """
    Compute per-player velocity (metres/second) via finite differences.
    Fills NaN at sequence boundaries with 0.
    """
    df = df.sort_values(["player", "frame"]).copy()
    df["vx"] = df.groupby("player")["x"].diff() * FPS
    df["vy"] = df.groupby("player")["y"].diff() * FPS
    df[["vx", "vy"]] = df[["vx", "vy"]].fillna(0.0)
    return df


home_tracking = add_velocities(home_tracking)
away_tracking = add_velocities(away_tracking)


def get_frame(home_df: pd.DataFrame, away_df: pd.DataFrame, frame_id: int):
    """Return (home_positions, away_positions, home_velocities, away_velocities) for one frame."""
    h = home_df[home_df["frame"] == frame_id].sort_values("jersey")
    a = away_df[away_df["frame"] == frame_id].sort_values("jersey")
    return (
        h[["x", "y"]].values,
        a[["x", "y"]].values,
        h[["vx", "vy"]].values,
        a[["vx", "vy"]].values,
        h["player"].values,
        a["player"].values,
    )


# Pick the frame of the first PASS event
first_pass  = events[events["type"] == "PASS"].iloc[0]
FRAME_ID    = int(first_pass["start_frame"])
print(f"Using frame {FRAME_ID}  (t={first_pass['start_time_s']:.2f}s)")
print(f"Event: {first_pass['team']} {first_pass['type']} from {first_pass['from']} to {first_pass['to']}")

home_pos, away_pos, home_vel, away_vel, home_players, away_players = get_frame(
    home_tracking, away_tracking, FRAME_ID
)
print(f"Home players in frame: {len(home_pos)}, Away players: {len(away_pos)}")

## 3. Build & Enrich the Graph

In [ ]:
# Build graph — positions already in metres, use coord_system="tracab" to skip rescaling
graph = build_graph_from_tracking_frame(
    home_positions  = home_pos,
    away_positions  = away_pos,
    home_velocities = home_vel,
    away_velocities = away_vel,
    coord_system    = "tracab",        # already in metres
    strategy        = EdgeStrategy.DELAUNAY,
    label           = 1.0,             # placeholder: pass success
)

graph = enrich_graph(graph, attacking_right=True, pressure_radius=5.0)

print("Graph summary")
print(f"  Nodes      : {graph.num_nodes}")
print(f"  Edges      : {graph.num_edges}")
print(f"  Node feats : {graph.x.shape[1]}  {list(graph.x.shape)}")
print(f"  Edge feats : {graph.edge_attr.shape[1]}  {list(graph.edge_attr.shape)}")
print(f"  Label      : {graph.y.item()}")
print()
print("Node feature columns: [x, y, vx, vy, team, dist_atk_goal, dist_def_goal, angle_atk, pressure]")
print(graph.x[:3])

## 4. Visualise Graph on Pitch

In [ ]:
def plot_player_graph(graph, home_players, away_players, title=""):
    """
    Draw the player graph overlaid on a pitch.
    Blue = home, Red = away.
    Edge opacity scales with 1/(distance+1) so close players have darker edges.
    """
    x_np        = graph.x.numpy()
    edge_index  = graph.edge_index.numpy()
    edge_attr   = graph.edge_attr.numpy()
    positions   = x_np[:, :2]           # metres
    teams       = x_np[:, 4].astype(int) # 0=home, 1=away
    pressure    = x_np[:, 8]            # pressure index

    all_players = np.concatenate([home_players, away_players])

    pitch = Pitch(pitch_type="metricasports", pitch_length=105, pitch_width=68,
                  pitch_color="grass", line_color="white", stripe=True)
    fig, ax = pitch.draw(figsize=(14, 9))
    fig.patch.set_facecolor("#1a1a2e")

    # Draw edges
    distances = edge_attr[:, 0]          # first edge feature is distance
    max_dist  = distances.max() if len(distances) else 1.0
    for idx in range(edge_index.shape[1]):
        src, dst = edge_index[0, idx], edge_index[1, idx]
        if src >= dst:                   # draw each undirected edge once
            continue
        alpha = float(1.0 - distances[idx] / (max_dist + 1e-6)) * 0.6 + 0.1
        same_team = edge_attr[idx, 3] == 1.0
        color = "#4fc3f7" if same_team else "#ef9a9a"
        ax.plot(
            [positions[src, 0], positions[dst, 0]],
            [positions[src, 1], positions[dst, 1]],
            color=color, linewidth=0.8, alpha=alpha, zorder=2
        )

    # Draw nodes
    colors = ["#1565c0" if t == 0 else "#b71c1c" for t in teams]
    sizes  = 200 + pressure * 80        # bigger node = more pressure
    ax.scatter(
        positions[:, 0], positions[:, 1],
        c=colors, s=sizes, zorder=3, edgecolors="white", linewidths=1.2
    )

    # Label jersey numbers
    for i, (px, py) in enumerate(positions):
        label = all_players[i].replace("Player", "") if i < len(all_players) else str(i)
        ax.text(px, py, label, ha="center", va="center",
                fontsize=6, color="white", fontweight="bold", zorder=4)

    # Legend
    home_patch  = mpatches.Patch(color="#1565c0", label="Home")
    away_patch  = mpatches.Patch(color="#b71c1c", label="Away")
    same_line   = plt.Line2D([0], [0], color="#4fc3f7", lw=1.5, label="Same-team edge")
    cross_line  = plt.Line2D([0], [0], color="#ef9a9a", lw=1.5, label="Cross-team edge")
    ax.legend(handles=[home_patch, away_patch, same_line, cross_line],
              loc="upper right", framealpha=0.7, fontsize=8)

    ax.set_title(title, color="white", fontsize=13, pad=10)
    plt.tight_layout()
    return fig


fig = plot_player_graph(
    graph, home_players, away_players,
    title=f"Frame {FRAME_ID} — Delaunay Graph  |  {first_pass['team']} PASS  {first_pass['from']} → {first_pass['to']}"
)
plt.show()

## 5. Compare Edge Strategies Side-by-Side

In [ ]:
strategies = [
    (EdgeStrategy.DELAUNAY,  "Delaunay"),
    (EdgeStrategy.PROXIMITY, "Proximity (r=12m)"),
    (EdgeStrategy.KNN,       "k-NN (k=4)"),
]

fig, axes = plt.subplots(1, 3, figsize=(21, 7))
fig.patch.set_facecolor("#1a1a2e")

for ax, (strategy, label) in zip(axes, strategies):
    g = build_graph_from_tracking_frame(
        home_pos, away_pos, home_vel, away_vel,
        coord_system="tracab", strategy=strategy, radius=12.0, k=4
    )

    x_np       = g.x.numpy()
    ei         = g.edge_index.numpy()
    positions  = x_np[:, :2]
    teams      = x_np[:, 4].astype(int)
    colors     = ["#1565c0" if t == 0 else "#b71c1c" for t in teams]

    pitch = Pitch(pitch_type="metricasports", pitch_length=105, pitch_width=68,
                  pitch_color="grass", line_color="white", stripe=True)
    pitch.draw(ax=ax)

    for idx in range(ei.shape[1]):
        src, dst = ei[0, idx], ei[1, idx]
        if src >= dst:
            continue
        same = teams[src] == teams[dst]
        color = "#4fc3f7" if same else "#ef9a9a"
        ax.plot([positions[src,0], positions[dst,0]],
                [positions[src,1], positions[dst,1]],
                color=color, lw=0.9, alpha=0.5, zorder=2)

    ax.scatter(positions[:, 0], positions[:, 1],
               c=colors, s=120, zorder=3, edgecolors="white", linewidths=1)

    ax.set_title(f"{label}  ({g.num_edges} edges)",
                 color="white", fontsize=11, pad=6)

plt.suptitle(f"Frame {FRAME_ID} — Edge strategy comparison",
             color="white", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 6. Build a Dataset of Graphs Aligned to Pass Events

In [ ]:
pass_events = events[events["type"] == "PASS"].reset_index(drop=True)
print(f"Total PASS events: {len(pass_events)}")
pass_events[["team", "subtype", "from", "to", "start_frame", "end_frame"]].head()


In [ ]:
from torch_geometric.data import Data

dataset: list[Data] = []
skipped = 0

for _, row in pass_events.iterrows():
    fid = int(row["start_frame"])

    h_pos, a_pos, h_vel, a_vel, h_pl, a_pl = get_frame(
        home_tracking, away_tracking, fid
    )

    # Skip frames where fewer than 10 players are tracked per side
    if len(h_pos) < 10 or len(a_pos) < 10:
        skipped += 1
        continue

    # Label: 1 = pass completed (has a 'to' player), 0 = intercepted / out
    label = 0.0 if pd.isna(row.get("to", float("nan"))) else 1.0

    g = build_graph_from_tracking_frame(
        h_pos, a_pos, h_vel, a_vel,
        coord_system="tracab",
        strategy=EdgeStrategy.DELAUNAY,
        label=label,
    )
    g = enrich_graph(g, attacking_right=(row["team"] == "Home"))
    g.frame_id = fid
    dataset.append(g)

print(f"Built {len(dataset)} graphs  ({skipped} skipped due to incomplete tracking)")
print(f"Successful passes: {sum(int(g.y.item()) for g in dataset)}  /  {len(dataset)}")
print(f"Node feature dim : {dataset[0].x.shape[1]}")
print(f"Edge feature dim : {dataset[0].edge_attr.shape[1]}")

In [ ]:
# Save the dataset for use in notebook 03
import os
os.makedirs("../../data/processed", exist_ok=True)
torch.save(dataset, "../../data/processed/metrica_game1_pass_graphs.pt")
print(f"Saved {len(dataset)} graphs to data/processed/metrica_game1_pass_graphs.pt")